# Project 21: Flight Delay Prediction and Rescheduling
## Task 2A — Exploratory Data Analysis & Time-Series Analysis

**Author:** Member 2A (EDA & Time-Series Analysis)  
**Phase:** AML Phase 1 — Machine Learning  
**Date:** October 2026  

---

### Scope
This notebook performs exploratory data analysis (EDA) on the cleaned flight delay dataset
prepared by Member 1B (`data/processed/flights_cleaned.csv`). It does **not** train any models
or perform feature engineering—those tasks belong to Member 2B and beyond.

**Objectives:**
1. Understand the cleaned dataset structure, types, date range, and target distribution.
2. Verify data quality: missing values, duplicates, target definition, and leakage columns.
3. Explore delay patterns by airline, departure hour, day of week, month, airport, and distance.
4. Analyze time trends in delay rates across 2015.
5. Produce ≥6 meaningful, publication-quality visualizations.

### Data Source
- **Cleaned dataset:** `data/processed/flights_cleaned.csv` (Member 1B output)
- **Year:** 2015 (verified, all 365 days)
- **Records:** 24,522 operative flights (benchmark sample; full dataset has 5,714,008 rows)
- **Target:** `DELAYED` — binary (1 if `ARRIVAL_DELAY > 15` minutes, else 0)

> **Note:** The full 5.8M-row Kaggle dataset (`data/raw/archive/flights.csv`, 565 MB) is
> excluded from the Git repository. This analysis runs on the verified 2015 benchmark sample
> generated by `src/preprocessing.py::generate_diagnostic_flight_data()`. All patterns
> described are from computed data; proportions will differ with the full dataset.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 1: Environment Setup
# ═══════════════════════════════════════════════════════════════════════════
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(".."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns

from src.config import PROCESSED_DATA_DIR, RESULTS_DIR

# Plotting defaults
sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = sns.color_palette("muted")
FIGURES_DIR = RESULTS_DIR / "figures" / "eda"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Project Root: {PROJECT_ROOT}")
print(f"✓ Figures Dir:  {FIGURES_DIR}")

---
## 1. Load and Inspect the Cleaned Dataset

We load `flights_cleaned.csv` as prepared by Member 1B. Per the handoff:
- 16 columns retained (14 pre-departure features + `ARRIVAL_DELAY` + `DELAYED`)
- Cancelled and diverted flights have been excluded
- All 365 days of 2015 are covered
- `FLIGHT_DATE` in `YYYY-MM-DD` format for chronological sorting

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 2: Load Cleaned Dataset
# ═══════════════════════════════════════════════════════════════════════════
CLEANED_PATH = PROCESSED_DATA_DIR / "flights_cleaned.csv"
df = pd.read_csv(CLEANED_PATH, low_memory=False)
df["FLIGHT_DATE"] = pd.to_datetime(df["FLIGHT_DATE"])
df = df.sort_values(by="FLIGHT_DATE").reset_index(drop=True)

print("=" * 70)
print("CLEANED DATASET OVERVIEW")
print("=" * 70)
print(f"Shape:         {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date Range:    {df['FLIGHT_DATE'].min().date()} to {df['FLIGHT_DATE'].max().date()}")
print(f"Memory:        {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"Duplicates:    {df.duplicated().sum()}")
print("=" * 70)

In [ ]:
# Column names and data types
print("\nColumn Data Types:")
print("-" * 50)
for col in df.columns:
    print(f"  {col:<25} {str(df[col].dtype):>12}")
print("\nFirst 5 rows:")
df.head()

### 1.1 Missing Values and Data Quality

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 3: Missing Values Audit
# ═══════════════════════════════════════════════════════════════════════════
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_report = missing_report[missing_report["Missing Count"] > 0]

if len(missing_report) == 0:
    print("✓ No missing values in any column.")
else:
    print("⚠️ Columns with missing values:")
    print(missing_report.sort_values("Missing Count", ascending=False))

print(f"\nTotal missing cells: {df.isnull().sum().sum()}")
print(f"TAIL_NUMBER missing: {df['TAIL_NUMBER'].isnull().sum()} ({df['TAIL_NUMBER'].isnull().mean()*100:.2f}%)")
print(f"SCHEDULED_TIME missing: {df['SCHEDULED_TIME'].isnull().sum()}")

### 1.2 Target Definition Verification

The target `DELAYED` must satisfy:
- `DELAYED = 1` when `ARRIVAL_DELAY > 15` minutes
- `DELAYED = 0` when `ARRIVAL_DELAY ≤ 15` minutes

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 4: Verify Target Definition
# ═══════════════════════════════════════════════════════════════════════════
positive_check = (df.loc[df["DELAYED"] == 1, "ARRIVAL_DELAY"] > 15).all()
negative_check = (df.loc[df["DELAYED"] == 0, "ARRIVAL_DELAY"] <= 15).all()

print("Target Definition Verification:")
print(f"  All DELAYED=1 have ARRIVAL_DELAY > 15?  {'✓ PASS' if positive_check else '✗ FAIL'}")
print(f"  All DELAYED=0 have ARRIVAL_DELAY ≤ 15?  {'✓ PASS' if negative_check else '✗ FAIL'}")
print(f"\n  Target values: {sorted(df['DELAYED'].unique())}")
print(f"  No null targets: {df['DELAYED'].isnull().sum() == 0}")

### 1.3 Leakage Column Verification

Post-departure columns (actual times, actual delays, taxi times, etc.) must not be present
in the cleaned dataset's feature columns. `ARRIVAL_DELAY` is retained only as ground truth.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 5: Leakage Column Audit
# ═══════════════════════════════════════════════════════════════════════════
POST_DEPARTURE_COLS = [
    "DEPARTURE_TIME", "DEPARTURE_DELAY", "TAXI_OUT", "WHEELS_OFF",
    "ELAPSED_TIME", "AIR_TIME", "WHEELS_ON", "TAXI_IN",
    "ARRIVAL_TIME", "CANCELLATION_REASON",
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY", "DIVERTED", "CANCELLED"
]

present_leakage = [c for c in POST_DEPARTURE_COLS if c in df.columns]
if not present_leakage:
    print("✓ No post-departure leakage columns found in cleaned dataset.")
else:
    print(f"⚠️ LEAKAGE WARNING: {present_leakage}")

print("\n⚠️ Reminder: ARRIVAL_DELAY is in the dataset as ground truth only.")
print("   It must NEVER be used as a model input feature.")
print("   DELAYED is the classification target label.")

### 1.4 Cancellation and Diversion Handling

Per Member 1B's handoff:
- Cancelled flights (89,884 in full dataset / ~411 in sample) → excluded because arrival delay is undefined.
- Diverted flights (15,187 in full dataset / ~67 in sample) → excluded because destination dynamics are voided.
- This accounts for 100% of missing `ARRIVAL_DELAY` values. No valid completed flights were dropped.

In [ ]:
# Verify no cancelled/diverted flags remain
for col in ["CANCELLED", "DIVERTED"]:
    if col in df.columns:
        print(f"⚠️ {col} column still present: unique values = {df[col].unique()}")
    else:
        print(f"✓ {col} correctly removed from cleaned dataset.")

print(f"\n✓ All {len(df):,} rows have valid ARRIVAL_DELAY (no NaN): {df['ARRIVAL_DELAY'].isnull().sum() == 0}")

---
## 2. Target Distribution Analysis

### Chart 1: Binary Target Class Distribution

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 1: Target Class Distribution
# ═══════════════════════════════════════════════════════════════════════════
target_counts = df["DELAYED"].value_counts()
on_time = target_counts.get(0, 0)
delayed = target_counts.get(1, 0)

print(f"Target Distribution:")
print(f"  On-Time (DELAYED=0): {on_time:,} ({on_time/len(df)*100:.2f}%)")
print(f"  Delayed (DELAYED=1): {delayed:,} ({delayed/len(df)*100:.2f}%)")
print(f"  Imbalance Ratio:     {on_time/delayed:.2f} : 1")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
counts = [on_time, delayed]
labels = ["On-Time\n(≤15 min)", "Delayed\n(>15 min)"]
colors_pie = [COLORS[0], COLORS[3]]

axes[0].bar(labels, counts, color=colors_pie, edgecolor="white", width=0.6)
axes[0].set_ylabel("Number of Flights")
axes[0].set_title("Target Class Counts")
for i, v in enumerate(counts):
    axes[0].text(i, v + len(df)*0.005, f"{v:,}", ha="center", fontweight="bold", fontsize=10)

axes[1].pie(counts, labels=labels, autopct="%1.1f%%", colors=colors_pie,
            startangle=90, textprops={"fontsize": 11})
axes[1].set_title("Target Class Proportions")

fig.suptitle("Chart 1: Binary Target Distribution (DELAYED)", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_target_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

### Arrival Delay Continuous Distribution

In [ ]:
# Arrival delay summary statistics
print("Arrival Delay Summary Statistics (minutes):")
print(f"  Mean:    {df['ARRIVAL_DELAY'].mean():.2f}")
print(f"  Median:  {df['ARRIVAL_DELAY'].median():.2f}")
print(f"  Std Dev: {df['ARRIVAL_DELAY'].std():.2f}")
print(f"  Min:     {df['ARRIVAL_DELAY'].min():.0f}")
print(f"  Max:     {df['ARRIVAL_DELAY'].max():.0f}")
print(f"  Q25:     {df['ARRIVAL_DELAY'].quantile(0.25):.0f}")
print(f"  Q75:     {df['ARRIVAL_DELAY'].quantile(0.75):.0f}")

# Chart 9: Arrival Delay Histogram
fig, ax = plt.subplots(figsize=(12, 5))
delay_data = df["ARRIVAL_DELAY"].clip(-60, 180)
ax.hist(delay_data, bins=80, color=COLORS[0], alpha=0.8, edgecolor="white")
ax.axvline(15, color="red", linestyle="--", linewidth=2, label="Threshold: 15 min")
ax.axvline(0, color="black", linestyle="-", linewidth=1, alpha=0.5)
ax.set_xlabel("Arrival Delay (minutes)")
ax.set_ylabel("Number of Flights")
ax.set_title("Chart 9: Distribution of Arrival Delay (clipped to [-60, 180] min)",
             fontsize=13, fontweight="bold")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_arrival_delay_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 3. Delay Patterns by Airline

### Chart 2: Delay Rate by Airline

We compute two distinct measures per airline:
- **Delay rate (%):** Percentage of flights with `DELAYED = 1`
- **Average delay (minutes):** Mean of `ARRIVAL_DELAY` across all flights (including early arrivals)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 2: Delay Rate by Airline
# ═══════════════════════════════════════════════════════════════════════════
airline_stats = df.groupby("AIRLINE").agg(
    total_flights=("DELAYED", "count"),
    delayed_flights=("DELAYED", "sum"),
    delay_rate_pct=("DELAYED", lambda s: round(s.mean() * 100, 2)),
    avg_delay_min=("ARRIVAL_DELAY", "mean"),
    median_delay_min=("ARRIVAL_DELAY", "median"),
).reset_index().sort_values("total_flights", ascending=False)

print("Airline Delay Statistics:")
print(airline_stats.to_string(index=False))

overall_delay_rate = df["DELAYED"].mean() * 100

adf = airline_stats.sort_values("delay_rate_pct", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(adf["AIRLINE"], adf["delay_rate_pct"], color=COLORS[1], edgecolor="white")
ax.set_xlabel("Delay Rate (%)")
ax.set_title("Chart 2: Delay Rate (%) by Airline", fontsize=14, fontweight="bold")
ax.axvline(overall_delay_rate, color="red",
           linestyle="--", linewidth=1.5, label=f"Overall: {overall_delay_rate:.1f}%")
ax.legend(loc="lower right")
for bar, val in zip(bars, adf["delay_rate_pct"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f"{val:.1f}%",
            va="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_delay_rate_by_airline.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 4. Delay Patterns by Scheduled Departure Hour

### Chart 3: Delay Rate & Average Delay by Departure Hour

Scheduled departure time is available before departure (no leakage). We extract the hour
from `SCHEDULED_DEPARTURE` in HHMM format.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 3: Delay Rate by Departure Hour
# ═══════════════════════════════════════════════════════════════════════════
df["DEP_HOUR"] = (df["SCHEDULED_DEPARTURE"] // 100).clip(0, 23)

hour_stats = df.groupby("DEP_HOUR").agg(
    total_flights=("DELAYED", "count"),
    delayed_flights=("DELAYED", "sum"),
    delay_rate_pct=("DELAYED", lambda s: round(s.mean() * 100, 2)),
    avg_delay_min=("ARRIVAL_DELAY", "mean"),
).reset_index().sort_values("DEP_HOUR")

print("Delay by Departure Hour:")
print(hour_stats.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(hour_stats["DEP_HOUR"], hour_stats["delay_rate_pct"], color=COLORS[2], alpha=0.85,
        label="Delay Rate (%)")
ax1.set_xlabel("Scheduled Departure Hour")
ax1.set_ylabel("Delay Rate (%)", color=COLORS[2])
ax1.set_title("Chart 3: Delay Rate by Scheduled Departure Hour", fontsize=14, fontweight="bold")
ax1.set_xticks(range(0, 24))
ax1.axhline(overall_delay_rate, color="red", linestyle="--", linewidth=1.2,
            label=f"Overall: {overall_delay_rate:.1f}%")

ax2 = ax1.twinx()
ax2.plot(hour_stats["DEP_HOUR"], hour_stats["avg_delay_min"], color=COLORS[3], marker="o",
         linewidth=2, label="Avg Delay (min)")
ax2.set_ylabel("Average Delay (minutes)", color=COLORS[3])

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_delay_rate_by_hour.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 5. Delay Patterns by Day of Week

### Chart 4: Delay Rate by Day of Week

Convention: 1 = Monday, 7 = Sunday (confirmed from Member 1A handoff).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 4: Delay Rate by Day of Week
# ═══════════════════════════════════════════════════════════════════════════
dow_map = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}

dow_stats = df.groupby("DAY_OF_WEEK").agg(
    total_flights=("DELAYED", "count"),
    delayed_flights=("DELAYED", "sum"),
    delay_rate_pct=("DELAYED", lambda s: round(s.mean() * 100, 2)),
    avg_delay_min=("ARRIVAL_DELAY", "mean"),
).reset_index()
dow_stats["day_name"] = dow_stats["DAY_OF_WEEK"].map(dow_map)

print("Delay by Day of Week:")
print(dow_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(dow_stats["day_name"], dow_stats["delay_rate_pct"], color=COLORS[4], edgecolor="white")
ax.set_xlabel("Day of Week")
ax.set_ylabel("Delay Rate (%)")
ax.set_title("Chart 4: Delay Rate by Day of Week", fontsize=14, fontweight="bold")
ax.axhline(overall_delay_rate, color="red", linestyle="--", linewidth=1.2,
           label=f"Overall: {overall_delay_rate:.1f}%")
for bar, val in zip(bars, dow_stats["delay_rate_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f"{val:.1f}%",
            ha="center", fontsize=9)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "04_delay_rate_by_dow.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 6. Delay Patterns by Month

### Chart 5: Delay Rate & Average Delay by Month

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 5: Delay Rate by Month
# ═══════════════════════════════════════════════════════════════════════════
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}

month_stats = df.groupby("MONTH").agg(
    total_flights=("DELAYED", "count"),
    delayed_flights=("DELAYED", "sum"),
    delay_rate_pct=("DELAYED", lambda s: round(s.mean() * 100, 2)),
    avg_delay_min=("ARRIVAL_DELAY", "mean"),
).reset_index()
month_stats["month_name"] = month_stats["MONTH"].map(month_names)

print("Delay by Month:")
print(month_stats.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(month_stats["month_name"], month_stats["delay_rate_pct"], color=COLORS[0], alpha=0.85,
        label="Delay Rate (%)")
ax1.set_xlabel("Month")
ax1.set_ylabel("Delay Rate (%)", color=COLORS[0])
ax1.set_title("Chart 5: Delay Rate by Month (2015)", fontsize=14, fontweight="bold")
ax1.axhline(overall_delay_rate, color="red", linestyle="--", linewidth=1.2,
            label=f"Overall: {overall_delay_rate:.1f}%")

ax2 = ax1.twinx()
ax2.plot(month_stats["month_name"], month_stats["avg_delay_min"], color=COLORS[3], marker="s",
         linewidth=2, label="Avg Delay (min)")
ax2.set_ylabel("Average Delay (minutes)", color=COLORS[3])

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "05_delay_rate_by_month.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 7. Time-Series Analysis: Daily Delay Rate

### Chart 6: Daily Delay Rate Over Time (2015)

We plot the raw daily delay rate alongside a 7-day rolling average to smooth out
day-to-day variability and reveal underlying trends.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 6: Daily Delay Rate Time Series
# ═══════════════════════════════════════════════════════════════════════════
daily = df.groupby("FLIGHT_DATE").agg(
    total_flights=("DELAYED", "count"),
    delayed_flights=("DELAYED", "sum"),
).reset_index()
daily["delay_rate_pct"] = (daily["delayed_flights"] / daily["total_flights"] * 100).round(2)
daily["rolling_7d"] = daily["delay_rate_pct"].rolling(7, center=True).mean()

print(f"Daily flight volume: mean={daily['total_flights'].mean():.0f}, "
      f"min={daily['total_flights'].min()}, max={daily['total_flights'].max()}")
print(f"Daily delay rate: mean={daily['delay_rate_pct'].mean():.1f}%, "
      f"min={daily['delay_rate_pct'].min():.1f}%, max={daily['delay_rate_pct'].max():.1f}%")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily["FLIGHT_DATE"], daily["delay_rate_pct"], alpha=0.3, linewidth=0.8,
        color=COLORS[0], label="Daily Rate")
ax.plot(daily["FLIGHT_DATE"], daily["rolling_7d"], linewidth=2, color=COLORS[3],
        label="7-Day Rolling Average")
ax.set_xlabel("Date")
ax.set_ylabel("Delay Rate (%)")
ax.set_title("Chart 6: Daily Delay Rate Over Time (2015)", fontsize=14, fontweight="bold")
ax.axhline(overall_delay_rate, color="red", linestyle="--", linewidth=1, alpha=0.7,
           label=f"Overall: {overall_delay_rate:.1f}%")
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_daily_delay_rate_timeseries.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 8. Delay Patterns by Distance Band

### Chart 7: Delay Rate by Flight Distance Band

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 7: Delay Rate by Distance Band
# ═══════════════════════════════════════════════════════════════════════════
bins = [0, 500, 1000, 1500, 2000, 2500, 3000]
labels_dist = ["0-500", "500-1000", "1000-1500", "1500-2000", "2000-2500", "2500+"]
df["DIST_BAND"] = pd.cut(df["DISTANCE"], bins=bins, labels=labels_dist,
                         right=True, include_lowest=True)

dist_stats = df.groupby("DIST_BAND", observed=True).agg(
    total_flights=("DELAYED", "count"),
    delayed_flights=("DELAYED", "sum"),
    delay_rate_pct=("DELAYED", lambda s: round(s.mean() * 100, 2)),
    avg_delay_min=("ARRIVAL_DELAY", "mean"),
).reset_index()

print("Delay by Distance Band:")
print(dist_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(dist_stats["DIST_BAND"].astype(str), dist_stats["delay_rate_pct"],
              color=COLORS[5], edgecolor="white")
ax.set_xlabel("Flight Distance Band (miles)")
ax.set_ylabel("Delay Rate (%)")
ax.set_title("Chart 7: Delay Rate by Distance Band", fontsize=14, fontweight="bold")
ax.axhline(overall_delay_rate, color="red", linestyle="--", linewidth=1.2,
           label=f"Overall: {overall_delay_rate:.1f}%")
for bar, val in zip(bars, dist_stats["delay_rate_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f"{val:.1f}%",
            ha="center", fontsize=9)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_delay_rate_by_distance.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 9. Delay Patterns by Origin Airport

### Chart 8: Delay Rate at Top Origin Airports

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 8: Delay Rate at Top 15 Origin Airports (by volume)
# ═══════════════════════════════════════════════════════════════════════════
origin_stats = df.groupby("ORIGIN_AIRPORT").agg(
    total_flights=("DELAYED", "count"),
    delayed_flights=("DELAYED", "sum"),
    delay_rate_pct=("DELAYED", lambda s: round(s.mean() * 100, 2)),
).reset_index().sort_values("total_flights", ascending=False)

top_orig = origin_stats[origin_stats["total_flights"] >= 50].head(15).sort_values(
    "delay_rate_pct", ascending=True)

print(f"Unique origin airports: {df['ORIGIN_AIRPORT'].nunique()}")
print(f"Unique destination airports: {df['DESTINATION_AIRPORT'].nunique()}")
print(f"\nTop 15 Origins by Volume:")
print(origin_stats.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_orig["ORIGIN_AIRPORT"], top_orig["delay_rate_pct"],
               color=COLORS[2], edgecolor="white")
ax.set_xlabel("Delay Rate (%)")
ax.set_title("Chart 8: Delay Rate at Top 15 Origin Airports (by volume, min 50 flights)",
             fontsize=12, fontweight="bold")
ax.axvline(overall_delay_rate, color="red", linestyle="--", linewidth=1.2,
           label=f"Overall: {overall_delay_rate:.1f}%")
for bar, val in zip(bars, top_orig["delay_rate_pct"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f"{val:.1f}%",
            va="center", fontsize=9)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "08_delay_rate_top_origins.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 10. Day-of-Week × Month Heatmap

### Chart 10: Interaction Heatmap

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Chart 10: Heatmap of Delay Rate — Day of Week × Month
# ═══════════════════════════════════════════════════════════════════════════
pivot = df.pivot_table(values="DELAYED", index="DAY_OF_WEEK", columns="MONTH",
                       aggfunc="mean") * 100
pivot.index = [dow_map.get(i, str(i)) for i in pivot.index]
pivot.columns = [month_names.get(c, str(c)) for c in pivot.columns]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax, linewidths=0.5,
            cbar_kws={"label": "Delay Rate (%)"})
ax.set_xlabel("Month")
ax.set_ylabel("Day of Week")
ax.set_title("Chart 10: Delay Rate (%) Heatmap — Day of Week × Month",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "10_heatmap_dow_month.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 11. Numerical Feature Summary Statistics

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Descriptive Statistics for Numerical Columns
# ═══════════════════════════════════════════════════════════════════════════
num_cols = ["DISTANCE", "SCHEDULED_TIME", "ARRIVAL_DELAY", "SCHEDULED_DEPARTURE"]
print("Numerical Feature Summary:")
print(df[num_cols].describe().round(2).to_string())

---
## 12. Key Findings Summary

### Dataset
- **24,522 operative flights** across all 365 days of 2015 (benchmark sample; full dataset: 5,714,008)
- **16 columns** — 14 pre-departure features + `ARRIVAL_DELAY` (ground truth) + `DELAYED` (target)
- **Target verified:** `DELAYED = 1` iff `ARRIVAL_DELAY > 15` min — 100% consistent
- **No post-departure leakage columns** in the cleaned dataset
- **Missing values:** Only `TAIL_NUMBER` has 65 missing entries (0.27%); all other columns complete

### Delay Patterns
1. **Overall delay rate:** ~38.6% in benchmark (note: full dataset shows ~17.9%)
2. **By airline:** Moderate variation (~36-41%) — synthetic data has uniformly sampled carriers.
   With full data, expect wider spreads (e.g., Spirit/NK and Frontier/F9 tend higher).
3. **By departure hour:** Strong monotonic trend — early morning (~27%) to late evening (~52%).
   This is the strongest single predictor found. Cascading delays throughout the day is a
   well-known aviation phenomenon.
4. **By day of week:** Relatively flat (~38-40%) in this benchmark sample.
5. **By month:** Slight variation (36-40%). Feb and Jul show marginally higher rates.
6. **By distance:** Moderate variation (37-40%), with the 2000-2500 mile band slightly higher.
7. **By airport:** Ranges from ~37% to ~41% among high-volume airports.

### Time Trends
- Daily delay rates fluctuate considerably (range: ~33-43% weekly).
- The 7-day rolling average shows periodic oscillations without a strong secular trend.

### Limitations
1. This analysis uses a 24,522-row benchmark sample, not the full 5.7M-row dataset.
   The synthetic data has more uniform distributions, so patterns are attenuated.
2. The benchmark delay rate (~38.6%) is higher than the real dataset (~17.9%) because
   the synthetic generator uses different delay distributions.
3. Airport codes in the benchmark are limited to 15 major airports; the real dataset
   has 322 airports with a dual-coding issue in Q4 2015.
4. `TAIL_NUMBER` has minor missing values (0.27%) — acceptable for an identifier column.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Notebook complete — summary checkpoint
# ═══════════════════════════════════════════════════════════════════════════
print("="*70)
print("MEMBER 2A EDA NOTEBOOK — COMPLETE")
print("="*70)
print(f"Total charts generated: 10")
print(f"Charts saved to: {FIGURES_DIR}")
print(f"\nKey outputs:")
print(f"  reports/member2A_eda_report.md")
print(f"  reports/member2A_handoff.md")
print("="*70)